<a href="https://colab.research.google.com/github/YukinoshitaSherry/CSCI572-Information_Retrieval_And_Web_Search_Engines/blob/master/scCode_ds_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from torch.utils.data import Dataset, DataLoader
import scanpy as sc
import anndata

In [ ]:
#define
class HVAE(nn.Module):
    """Hierarchical Variational Autoencoder for multi-modal integration"""
    def __init__(self, input_dim, latent_dim=128):
        super().__init__()
        # Multi-modal encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(),
            nn.Linear(512, latent_dim*2)
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(),
            nn.Linear(512, input_dim)
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std

    def forward(self, x):
        h = self.encoder(x)
        mu, logvar = torch.chunk(h, 2, dim=1)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

In [ ]:
class PathwayGNN(nn.Module):
    """Pathway-aware Graph Neural Network with GAT"""
    def __init__(self, input_dim, hidden_dim=256):
        super().__init__()
        # Predefined gene-pathway adjacency matrix
        self.conv1 = GATConv(input_dim, hidden_dim, heads=3)
        self.conv2 = GATConv(hidden_dim*3, hidden_dim)

    def forward(self, x, edge_index):
        x = F.elu(self.conv1(x, edge_index))
        x = F.elu(self.conv2(x, edge_index))
        return x

In [ ]:
class TemporalRNN(nn.Module):
    """Temporal Dynamics Module with Bidirectional LSTM"""
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim,
                          batch_first=True, bidirectional=True)
        self.attention = nn.MultiheadAttention(hidden_dim*2, num_heads=4)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        attn_out, _ = self.attention(lstm_out, lstm_out, lstm_out)
        return attn_out

In [ ]:
class TCellResponsePredictor(nn.Module):
    """Integrated Multi-modal Prediction Model"""
    def __init__(self, input_dim, pathway_graph, device='cuda'):
        super().__init__()
        self.hvae = HVAE(input_dim)
        self.gnn = PathwayGNN(input_dim)
        self.rnn = TemporalRNN(input_dim)

        # Final prediction layer
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

        # Predefined pathway graph structure
        self.pathway_edge_index = pathway_graph.edge_index.to(device)

    def forward(self, x, time_series):
        # Multi-modal integration
        recon_x, mu, logvar = self.hvae(x)

        # Pathway analysis
        pathway_features = self.gnn(mu, self.pathway_edge_index)

        # Temporal dynamics
        temporal_features = self.rnn(time_series)

        # Feature fusion
        combined = torch.cat([pathway_features.mean(1),
                            temporal_features.mean(1)], dim=1)

        return self.classifier(combined), recon_x, mu, logvar

In [ ]:
from scipy import sparse
from sklearn.preprocessing import StandardScaler

class TCellDataset(Dataset):
    def __init__(self, adata_path):
        self.adata = sc.read(adata_path)
        self.preprocess()

    def preprocess(self):
        # Quality control
        sc.pp.filter_cells(self.adata, min_genes=200)
        sc.pp.filter_genes(self.adata, min_cells=30)

        # SCTransform normalization
        sc.experimental.pp.normalize_pearson_residuals(self.adata)

        # Batch correction using Harmony
        import harmonypy as hm
        ho = hm.run_ho(self.adata.obsm['X_pca'], self.adata.obs['batch'])
        self.adata.obsm['X_harmony'] = ho.Z_corr.T

        # Feature selection
        sc.pp.highly_variable_genes(self.adata, n_top_genes=2000)
        self.adata = self.adata[:, self.adata.var.highly_variable]

    def __getitem__(self, idx):
        # Extract multi-modal data
        expr = torch.tensor(self.adata.X[idx].toarray().squeeze())
        protein = torch.tensor(self.adata.obsm['protein'][idx])
        return expr, protein

    def __len__(self):
        return self.adata.n_obs

In [ ]:
import mlflow
import optuna
from sklearn.metrics import roc_auc_score

def train_model(config):
    # Initialize dataset
    dataset = TCellDataset("kang_normalized_hvg.h5ad")
    loader = DataLoader(dataset, batch_size=512, shuffle=True)

    # Initialize model
    model = TCellResponsePredictor(input_dim=2000,
                                  pathway_graph=load_pathway_graph())
    model.to(device)

    # Optimizer configuration
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'],
                                weight_decay=1e-5)

    # Custom loss function
    def loss_function(recon_x, x, mu, logvar, pred, target):
        BCE = F.binary_cross_entropy(recon_x, x, reduction='mean')
        KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        CLS = F.binary_cross_entropy_with_logits(pred, target)
        return BCE + KLD*0.1 + CLS

    # MLflow tracking
    with mlflow.start_run():
        mlflow.log_params(config)

        for epoch in range(config['epochs']):
            model.train()
            total_loss = 0

            for batch in loader:
                expr, protein = batch
                expr = expr.to(device)

                # Multi-modal input concatenation
                inputs = torch.cat([expr, protein], dim=1)

                optimizer.zero_grad()
                pred, recon, mu, logvar = model(inputs)
                loss = loss_function(recon, inputs, mu, logvar,
                                   pred, target_labels)

                loss.backward()
                optimizer.step()
                total_loss += loss.item()

            # Validation step
            model.eval()
            val_pred, val_labels = evaluate(model, val_loader)
            auc = roc_auc_score(val_labels, val_pred)

            mlflow.log_metrics({
                'train_loss': total_loss/len(loader),
                'val_auc': auc
            }, step=epoch)

In [ ]:
def evaluate(model, loader):
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in loader:
            inputs, labels = batch
            inputs = inputs.to(device)
            preds = model(inputs)[0].sigmoid().cpu()
            all_preds.append(preds)
            all_labels.append(labels)
    return torch.cat(all_preds), torch.cat(all_labels)

In [ ]:
def hyperparameter_optimization():
    study = optuna.create_study(direction='maximize')
    study.optimize(lambda trial: {
        'lr': trial.suggest_loguniform('lr', 1e-5, 1e-3),
        'batch_size': trial.suggest_categorical('batch_size', [256, 512]),
        'latent_dim': trial.suggest_int('latent_dim', 64, 256)
    }, n_trials=50)

    return study.best_params

In [ ]:
#biological Interpreter analysis
import shap
import numpy as np

def interpret_model(model, background_data):
    # SHAP analysis
    explainer = shap.DeepExplainer(model, background_data)
    shap_values = explainer.shap_values(test_samples)

    # Pathway importance analysis
    pathway_importance = calculate_pathway_importance(model.gnn)

    # Visualization
    plot_shap_summary(shap_values)
    plot_pathway_network(pathway_importance)

def calculate_pathway_importance(gnn):
    # Extract GAT attention weights
    edge_weights = gnn.conv1.att_scores.detach().cpu().numpy()
    pathway_scores = np.mean(edge_weights, axis=(0,1))
    return dict(zip(pathway_names, pathway_scores))

In [ ]:
import streamlit as st
from io import StringIO

@st.cache_resource
def load_production_model():
    return torch.load('best_model.pth')

def main():
    st.title("T-cell Response Prediction System")
    uploaded_file = st.file_uploader("Upload single-cell data (h5ad format)")

    if uploaded_file:
        adata = sc.read(StringIO(uploaded_file.getvalue().decode()))
        preprocessed = preprocess_data(adata)

        if st.button("Run Prediction"):
            with st.spinner('Running inference...'):
                predictions = model.predict(preprocessed)
                plot_predictions(predictions)

if __name__ == '__main__':
    main()